# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata and inspect name and description
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset title: ", metadata.name)
print("Description: ", metadata.description)

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.
Here we print all available record sets and their details, followed by fields for a selected record set.

In [ ]:
# List the available record sets and their @id
record_sets = list(metadata.record_sets)
print("Available Record Sets:")
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")

# For the first RecordSet, list available fields, referencing by field `@id`
if record_sets:
    main_recordset = record_sets[0]
    print(f"\nFields for RecordSet '{main_recordset.name}' (@id: {main_recordset.id}):")
    for field in main_recordset.fields:
        column = field.column
        colid = column.id if column is not None else None
        print(f"- Field name: {field.name}, @id: {field.id}, column @id: {colid}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Here, we load the first available record set (by `@id`) discovered in the overview step.

In [ ]:
# We'll load each record set found in the schema into DataFrames, referenced by their @id
dataframes = {}
for record_set in record_sets:
    print(f"Loading records for record set: {record_set.name} (@id: {record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"Columns: {df.columns.tolist()}")

# Pick the first record set for demonstration
chosen_record_set_id = record_sets[0].id if record_sets else None
if chosen_record_set_id:
    print(f"Example rows for {chosen_record_set_id}:")
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We'll select a numeric field (such as molecular weight or abundance measurement) to filter, normalize, and analyze.

In [ ]:
# You may need to inspect columns to pick an appropriate numeric field (e.g. 'MW_kDa', 'abundance', etc)
# For demonstration, let's try typical proteome fields: e.g., 'MW_kDa', 'Abundance', 'Coverage_percent', etc.
df = dataframes[chosen_record_set_id]
print("Available columns for EDA:")
print(df.columns.tolist())

# Let's try using 'MW_kDa' as a numeric field if present, else fall back to a similar field
possible_numeric_fields = ['MW_kDa', 'Abundance', 'Coverage_percent', 'unique_peptides', 'peptide_count']
numeric_field = None
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field = field
        break

if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a key attribute (select a string/categorical field)
    possible_group_fields = ['description', 'modification', 'sample', 'accession']
    group_field = None
    for field in possible_group_fields:
        if field in df.columns:
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check the record set columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Let's plot the distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, plot group means
    if group_field:
        group_means = df.groupby(group_field)[numeric_field].mean().dropna()
        group_means = group_means.sort_values(ascending=False)[:15]  # plot top 15 for clarity
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field} (top 15)")
        plt.xticks(rotation=90)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the Mass Spectrometry dataset using Croissant metadata.
- Available record sets and their fields were reviewed using their `@id`s, and the first record set was extracted into a DataFrame for analysis.
- Exploratory data analysis showcased filtering, normalization, and visualization of a relevant numeric field (e.g., protein molecular weight or abundance).
- Further analysis can be achieved by selecting biologically meaningful fields for deeper investigation or by integrating with external proteomics resources.

Feel free to adapt this notebook for additional record sets, fields, or custom analyses as needed!